# LTX-Video: Scene 1 Generation (Kaggle T4)
This notebook evaluates LTX-Video on a 16GB VRAM constraint.

In [ ]:
!pip install -q diffusers transformers accelerate imageio[ffmpeg] xformers

In [ ]:
import torch
import time
import json
import gc
from diffusers import LTXPipeline
from diffusers.utils import export_to_video

def print_vram_usage():
    torch.cuda.empty_cache()
    gc.collect()
    allocated = torch.cuda.memory_allocated() / (1024**3)
    reserved = torch.cuda.memory_reserved() / (1024**3)
    max_allocated = torch.cuda.max_memory_allocated() / (1024**3)
    print(f"VRAM Max Allocated: {max_allocated:.2f} GB | Reserved: {reserved:.2f} GB")
    return max(max_allocated, reserved)

print("Loading LTX-Video Pipeline...")
model_id = "Lightricks/LTX-Video"

# Load in bfloat16 for memory efficiency
pipe = LTXPipeline.from_pretrained(model_id, torch_dtype=torch.bfloat16)

# Memory Optimizations for Kaggle T4 (16GB)
pipe.enable_model_cpu_offload() # Crucial for 16GB VRAM

try:
    pipe.enable_xformers_memory_efficient_attention()
except Exception as e:
    print("Xformers not enabled, falling back.")

try:
    pipe.enable_vae_slicing()
    pipe.enable_vae_tiling()
except Exception as e:
    print(f"VAE optimization failed: {e}")

try:
    pipe.enable_attention_slicing()
except Exception as e:
    pass


In [ ]:
prompt = "Medium shot. A frail 6-year-old boy wakes up in a wooden bed, his eyes fluttering open. A small 5-year-old girl shakes his arm vigorously. The rest of the family surrounds the bed, their faces filled with relief and joy. Cinematic, manhwa style."
negative_prompt = "worst quality, inconsistent, blurry, deformed"

print(f"Prompt: {prompt}")
print("Starting generation...")

start_time = time.time()

try:
    # Conservative settings: 512x512, 3 seconds (~73 frames at 24fps)
    video = pipe(
        prompt=prompt,
        negative_prompt=negative_prompt,
        width=512,
        height=512,
        num_frames=73,
        num_inference_steps=30,
    ).frames[0]
    
except RuntimeError as e:
    if 'out of memory' in str(e).lower():
        print("\nABORT: VRAM exceeded limit (OOM).")
        peak_vram = 16.0
        video = None
    else:
        raise e

end_time = time.time()
generation_time = end_time - start_time
peak_vram = print_vram_usage()

if peak_vram > 15.0:
    print("\nWARNING/ABORT: Peak VRAM exceeded 15GB!")

if video:
    output_path = "scene1.mp4"
    export_to_video(video, output_path, fps=24)
    print(f"\nSaved video to {output_path}")


In [ ]:
results = {
    "scene_id": "Scene 1",
    "model_name": "Lightricks/LTX-Video",
    "generation_time_seconds": generation_time,
    "peak_vram_gb": peak_vram,
    "resolution": "512x512",
    "frame_count": 73,
    "fps": 24,
    "status": "Success" if peak_vram <= 15.0 and video else "Aborted (OOM or VRAM > 15GB)"
}

with open("results.json", "w") as f:
    json.dump(results, f, indent=4)

print("Results logged to results.json")
